## Création d'un fichier JSON à partir d'un PDF pour RAG

Ce notebook vous guidera à travers les étapes pour extraire du texte d'un document PDF, le découper en morceaux (chunks), générer des 'embeddings' (représentations numériques) pour chaque morceau, et enfin sauvegarder ces données dans un fichier JSON, un format idéal pour des applications de *Retrieval Augmented Generation* (RAG).

### Étape 1 : Installation des bibliothèques nécessaires

Nous aurons besoin de quelques outils (bibliothèques Python) pour accomplir cette tâche :
*   `pypdf` : Pour lire et extraire le texte des fichiers PDF.
*   `sentence-transformers` : Pour générer des embeddings à partir du texte. C'est un type de modèle d'intelligence artificielle qui transforme le texte en une liste de nombres (un vecteur) qui capture la signification du texte.
*   `tqdm` : Pour afficher une barre de progression, ce qui est utile pour les opérations qui prennent du temps.

In [ ]:
# Installation des bibliothèques. Le '!' au début permet d'exécuter une commande de terminal dans Colab.
!pip install pypdf sentence-transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 24.2 MB/s eta 0:00:00


### Étape 2 : Chargement du fichier PDF

Maintenant, nous allons vous permettre de charger votre fichier PDF directement depuis votre ordinateur vers l'environnement Google Colab. Colab dispose d'une fonction `files.upload()` pour cela.

In [ ]:
from google.colab import files
import io

print("Veuillez sélectionner le fichier PDF que vous souhaitez charger.")
uploaded = files.upload()

# Nous allons stocker le nom du fichier pour une utilisation ultérieure.
for filename in uploaded.keys():
    pdf_filename = filename
    print(f"Fichier '{pdf_filename}' chargé avec succès.")

# Lire le contenu du fichier PDF chargé
pdf_file_content = io.BytesIO(uploaded[pdf_filename])

Veuillez sélectionner le fichier PDF que vous souhaitez charger.


Saving 200069409_reglement_20250626.pdf to 200069409_reglement_20250626.pdf
Fichier '200069409_reglement_20250626.pdf' chargé avec succès.


### Étape 3 : Lecture du PDF et extraction du texte

Nous allons utiliser `pypdf` pour ouvrir le fichier PDF que vous avez chargé. Ensuite, nous allons parcourir chaque page pour en extraire tout le texte. Nous garderons une trace du texte de chaque page, ainsi que du numéro de page, pour pouvoir les utiliser plus tard.

In [ ]:
import pypdf
from tqdm.notebook import tqdm

def extract_text_from_pdf(pdf_file_content):
    reader = pypdf.PdfReader(pdf_file_content)
    text_per_page = []
    for i, page in enumerate(tqdm(reader.pages, desc="Extraction du texte des pages")):
        text = page.extract_text()
        if text:
            text_per_page.append({"page_number": i + 1, "text": text})
    return text_per_page

all_pages_text = extract_text_from_pdf(pdf_file_content)

print(f"Texte extrait de {len(all_pages_text)} pages.")
# Afficher un aperçu du texte de la première page pour vérification
if all_pages_text:
    print("\nAperçu du texte de la première page :")
    print(all_pages_text[0]['text'][:500] + "...") # Affiche les 500 premiers caractères
else:
    print("Aucun texte n'a pu être extrait du PDF.")

Extraction du texte des pages:   0%|          | 0/251 [00:00<?, ?it/s]

Texte extrait de 251 pages.

Aperçu du texte de la première page :
Plan Local d’Urbanisme 
intercommunal (PLUi)
Règlement écrit
Dossier approuvé
Vu pour être annexé à la délibération 
d’approbation du PLUi en date du 
26/06/2025...


### Étape 4 : Découpage du texte en morceaux (chunks) pour RAG

Pour des applications RAG (Retrieval Augmented Generation), il est souvent préférable de ne pas utiliser le texte complet d'une page, car il peut être trop long. Au lieu de cela, nous allons découper le texte en morceaux plus petits et gérables (appelés 'chunks').

Chaque chunk devrait être suffisamment petit pour être sémantiquement cohérent (il doit avoir un sens par lui-même), mais aussi contenir suffisamment de contexte pour être utile. Nous utiliserons une stratégie simple de découpage par taille de caractères, avec un chevauchement pour nous assurer que le contexte n'est pas perdu entre deux chunks.

In [ ]:
import re

def split_into_sentences(text):
    """
    Découpe le texte en phrases complètes.
    Les phrases ne sont jamais coupées entre deux chunks.
    """
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [sentence.strip() for sentence in sentences if sentence.strip()]


def chunk_text(text_per_page, chunk_size=800, chunk_overlap=150):
    chunks = []

    for page_data in tqdm(text_per_page, desc="Découpage du texte en chunks"):
        page_text = page_data['text']
        page_number = page_data['page_number']

        # Nettoyage des espaces et sauts de ligne
        page_text = ' '.join(page_text.split())

        # Découpage en phrases
        sentences = split_into_sentences(page_text)

        current_chunk = []
        current_length = 0

        for sentence in sentences:

            sentence_length = len(sentence)

            # Ajouter la phrase si elle tient dans le chunk actuel
            if current_length + sentence_length <= chunk_size:
                current_chunk.append(sentence)
                current_length += sentence_length + 1

            else:
                # Sauvegarde du chunk terminé
                if current_chunk:
                    chunks.append({
                        "document_name": pdf_filename,
                        "page_number": page_number,
                        "chunk_text": " ".join(current_chunk)
                    })

                # Création du chevauchement avec les dernières phrases
                overlap_sentences = []
                overlap_length = 0

                for previous_sentence in reversed(current_chunk):
                    if overlap_length + len(previous_sentence) <= chunk_overlap:
                        overlap_sentences.insert(0, previous_sentence)
                        overlap_length += len(previous_sentence)
                    else:
                        break

                # Nouveau chunk avec contexte conservé
                current_chunk = overlap_sentences + [sentence]
                current_length = sum(len(s) for s in current_chunk)

        # Ajout du dernier chunk de la page
        if current_chunk:
            chunks.append({
                "document_name": pdf_filename,
                "page_number": page_number,
                "chunk_text": " ".join(current_chunk)
            })


    # Suppression des éventuels doublons
    final_chunks = []
    seen_chunks = set()

    for chunk_data in chunks:
        if chunk_data["chunk_text"] not in seen_chunks:
            final_chunks.append(chunk_data)
            seen_chunks.add(chunk_data["chunk_text"])

    return final_chunks


# Paramètres de découpage
CHUNK_SIZE = 800       # Taille approximative d'un chunk en caractères
CHUNK_OVERLAP = 150    # Nombre de caractères conservés entre deux chunks

text_chunks = chunk_text(
    all_pages_text,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)


print(f"Nombre total de chunks créés : {len(text_chunks)}")


# Affichage d'un aperçu
if text_chunks:
    print("\nAperçu des 3 premiers chunks :")
    for i, chunk in enumerate(text_chunks[:3]):
        print(f"--- Chunk {i+1} (Page {chunk['page_number']}) ---")
        print(chunk['chunk_text'])
else:
    print("Aucun chunk n'a pu être créé.")

Découpage du texte en chunks:   0%|          | 0/251 [00:00<?, ?it/s]

Nombre total de chunks créés : 826

Aperçu des 3 premiers chunks :
--- Chunk 1 (Page 1) ---
Plan Local d’Urbanisme intercommunal (PLUi) Règlement écrit Dossier approuvé Vu pour être annexé à la délibération d’approbation du PLUi en date du 26/06/2025
--- Chunk 2 (Page 2) ---
SOMMAIRE MODE D’EMPLOI DU REGLEMENT ................................................................................... 4 Articulation avec les orientations d’aménagement et de programmation et les annexes . 6 Le règlement dans les grandes lignes .............................................................................. 6 Division du territoire en zones .......................................................................................... 7 Description des destinations et sous-destinations ........................................................... 8 Lexique ........................................................................................................................... 13 DISPOSITIONS GENERALES A

### Étape 5 : Génération des Embeddings pour chaque chunk

Les *embeddings* sont des représentations numériques du texte sous forme de vecteurs (listes de nombres). Ces vecteurs capturent la signification sémantique des mots et des phrases, ce qui signifie que des textes ayant un sens similaire auront des vecteurs proches dans l'espace multidimensionnel. C'est essentiel pour la recherche de pertinence dans les systèmes RAG.

Nous utiliserons un modèle pré-entraîné de la bibliothèque `sentence-transformers` pour transformer chaque chunk de texte en son vecteur embedding correspondant. Pour un usage gratuit et généraliste, un modèle comme `all-MiniLM-L6-v2` est un bon choix car il est efficace et performant.

In [ ]:
from sentence_transformers import SentenceTransformer

print("Chargement du modèle d'embeddings. Cela peut prendre un instant...")
# Télécharge et charge un modèle d'embeddings pré-entraîné. 'all-MiniLM-L6-v2' est un bon choix léger et performant.
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Modèle chargé avec succès.")

# Générer les embeddings pour tous les chunks
# Nous allons ajouter l'embedding à chaque dictionnaire de chunk
for chunk_data in tqdm(text_chunks, desc="Génération des embeddings"):
    # Le modèle encode le texte et retourne un vecteur NumPy. Nous le convertirons en liste pour le JSON.
    embedding = model.encode(chunk_data['chunk_text']).tolist()
    chunk_data['embedding'] = embedding

print(f"Embeddings générés pour {len(text_chunks)} chunks.")

# Afficher un aperçu du premier chunk avec son embedding (seulement les premières valeurs pour ne pas surcharger l'affichage)
if text_chunks:
    print("\nAperçu du premier chunk avec son embedding :")
    print(f"Document : {text_chunks[0]['document_name']}")
    print(f"Page : {text_chunks[0]['page_number']}")
    print(f"Texte du chunk : {text_chunks[0]['chunk_text'][:200]}...")
    print(f"Embedding (premières 5 valeurs) : {text_chunks[0]['embedding'][:5]}...")
else:
    print("Aucun chunk avec embedding à afficher.")

Chargement du modèle d'embeddings. Cela peut prendre un instant...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modèle chargé avec succès.


Génération des embeddings:   0%|          | 0/826 [00:00<?, ?it/s]

Embeddings générés pour 826 chunks.

Aperçu du premier chunk avec son embedding :
Document : 200069409_reglement_20250626.pdf
Page : 1
Texte du chunk : Plan Local d’Urbanisme intercommunal (PLUi) Règlement écrit Dossier approuvé Vu pour être annexé à la délibération d’approbation du PLUi en date du 26/06/2025...
Embedding (premières 5 valeurs) : [-0.025646371766924858, -0.011933842673897743, 0.09517055749893188, -0.09999708086252213, -0.0828898623585701]...


### Étape 6 : Création et téléchargement du fichier JSON final

Nous allons maintenant structurer toutes les données collectées (nom du document, numéro de page, texte du chunk, et son embedding) dans une liste d'objets JSON. Ensuite, nous écrirons cette liste dans un fichier JSON sur votre système Colab, et enfin, nous vous fournirons un lien pour télécharger ce fichier sur votre ordinateur.

In [ ]:
import json
from google.colab import files

# Préparer la structure finale des données
json_output = []
for chunk_data in text_chunks:
    json_output.append({
        "document_name": chunk_data["document_name"],
        "page_number": chunk_data["page_number"],
        "chunk_text": chunk_data["chunk_text"],
        "embedding": chunk_data["embedding"]
    })

# Définir le nom du fichier de sortie JSON
output_filename = f"{pdf_filename.replace('.pdf', '')}_chunks_with_embeddings.json"

# Écrire les données dans le fichier JSON
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(json_output, f, ensure_ascii=False, indent=4)

print(f"Fichier JSON '{output_filename}' créé avec succès.")

# Proposer le téléchargement du fichier
files.download(output_filename)

print("Processus terminé ! Le fichier JSON devrait être téléchargé sur votre ordinateur.")

Fichier JSON '200069409_reglement_20250626_chunks_with_embeddings.json' créé avec succès.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Processus terminé ! Le fichier JSON devrait être téléchargé sur votre ordinateur.


### Processus Terminé !

Félicitations ! Vous avez créé un script complet dans Google Colab pour :
1.  Charger un fichier PDF.
2.  Extraire le texte de chaque page.
3.  Découper le texte en chunks pour le RAG.
4.  Générer des embeddings pour chaque chunk.
5.  Créer un fichier JSON contenant toutes ces informations et le télécharger.

Ce fichier JSON est maintenant prêt à être utilisé dans des applications de RAG ou d'autres analyses basées sur les embeddings.

N'hésitez pas si vous avez d'autres questions ou si vous souhaitez explorer d'autres aspects !